<div style="background-color:#1B2A4A; padding:24px 28px; border-radius:6px; color:#FFFFFF; font-family:Arial, Helvetica, sans-serif;">
<h1 style="color:#FFFFFF; margin-top:0; margin-bottom:4px;">DiRAC Training Academy</h1>
<p style="color:#C7D0E0; margin-top:0; font-size:15px; letter-spacing:0.5px;">DISTRIBUTED RESEARCH UTILISING ADVANCED COMPUTING</p>
<hr style="border:none; border-top:1px solid #3D4E75; margin:16px 0;">
<h2 style="color:#FFFFFF; margin-bottom:4px;">Lesson 10: Using Python Effectively on HPC Systems</h2>
<p style="color:#C7D0E0; margin-bottom:0;">A Training Academy notebook</p>
</div>


## Overview

Python has become one of the most widely used programming languages in scientific research. It is used to prepare simulation input, automate workflows, analyse large datasets, visualise results and develop scientific applications.

Using Python effectively on a High Performance Computing (HPC) system involves more than simply writing Python programs. Researchers also need to understand how to manage software environments, organise reproducible workflows and write efficient code that makes good use of HPC resources.

In this lesson, you will bring together the skills developed throughout this course and learn how Python fits into a modern scientific computing workflow. You will also explore good programming practices that improve portability, reproducibility and performance across a range of UK HPC services, including DiRAC.

**Estimated study time:** 60 minutes

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain how Python is used throughout scientific HPC workflows.
- Create and manage Python virtual environments.
- Install and manage Python packages for research projects.
- Understand how Python applications are typically executed on HPC systems.
- Apply techniques for writing efficient scientific Python code.
- Follow good practice for developing reproducible research software.

> **Note for students — this lesson is a bit different.** So far, almost everything in this course has been a line of Python you could run directly in a notebook cell. This lesson is partly about things that happen **outside** Python entirely — managing environments and packages from a terminal. Where a code block is something you'd type into a **terminal**, not a notebook cell, it's clearly labelled. Where it's real runnable Python, it's in a normal code cell as usual. This lesson also pulls together everything from Lessons 6–9 (files, NumPy, pandas, Matplotlib) into complete working examples — it's the capstone of the course.


## Python in Scientific Computing

Python is rarely used in isolation. Instead, it supports every stage of a scientific workflow. Typical tasks include:

- Preparing simulation input.
- Automating repetitive tasks.
- Running analysis pipelines.
- Processing simulation outputs.
- Creating publication-quality figures.
- Building scientific workflows.

Many HPC applications are written in C, C++ or Fortran, with Python providing a flexible interface for preparing, analysing and visualising data.

## A Typical HPC Workflow

```
Prepare input data
       |
       v
Run simulation
       |
       v
Collect output files
       |
       v
Analyse data
       |
       v
Create figures
       |
       v
Publish results
```

The previous nine lessons have introduced each of these stages individually — Lesson 6 for files, Lesson 7 for numerical analysis, Lesson 8 for structured data, Lesson 9 for figures. This lesson demonstrates how they fit together.


## Managing Python Environments

Different research projects often require different versions of Python packages. Rather than installing packages globally, it is good practice to create an isolated Python environment for each project.

This helps to:

- avoid software conflicts;
- improve reproducibility;
- make projects easier to share;
- simplify software maintenance.

## Creating a Virtual Environment

> 🖥️ **Run these in a terminal, not in a notebook cell.** A Jupyter notebook cell runs inside one already-started Python process (the "kernel"). Creating and "activating" a virtual environment changes settings for a *new* terminal session — it has no lasting effect on a kernel that's already running, so these commands belong in a terminal window (or JupyterHub's Terminal app), not here.

Create a new environment:

```bash
python -m venv research_env
```

Activate it.

Linux and macOS:
```bash
source research_env/bin/activate
```

Windows:
```bash
research_env\Scripts\activate
```

Deactivate when finished:
```bash
deactivate
```

> **If you do want to use a specific virtual environment's packages *inside* a notebook**, the usual approach is to install `ipykernel` inside that environment and register it as a Jupyter kernel, then pick it from the **Kernel** menu in Jupyter — rather than trying to "activate" it from inside a running notebook cell. This is a bit beyond this lesson's scope, but worth knowing the right approach exists.

## Installing Python Packages

After activating the environment, in your terminal:

```bash
pip install numpy
pip install pandas
pip install matplotlib
```

Many HPC systems also provide centrally managed Python environments and scientific libraries through **software modules** (typically loaded with a `module load` command). Always check your local system's documentation before installing additional software — many sites prefer you use their provided, optimised builds of common libraries rather than installing your own.

> **A refinement on installing packages from inside a notebook:** earlier in this course, you may have used `!pip install numpy` to fix a `ModuleNotFoundError`. That generally works, but the safer command — one that's guaranteed to install into *this exact notebook's* kernel, even if your system has several Pythons installed — is `%pip install numpy` (using `%`, not `!`). Both do the same job most of the time; `%pip` is just the more reliable choice specifically inside a notebook.


## Writing Efficient Python

Python itself is an interpreted language, but many scientific libraries (like NumPy) perform their calculations using highly optimised compiled code underneath. Good Python programs therefore spend most of their time *inside* these libraries rather than executing Python loops directly.

### Prefer Vectorised Operations

Instead of:


In [ ]:
# Explicit loop version
data = [1, 2, 3, 4, 5]
result = [0] * len(data)

for i in range(len(data)):
    result[i] = data[i] * 2

print(result)

Use, when `data` is a NumPy array:


In [ ]:
import numpy as np

data = np.array([1, 2, 3, 4, 5])
result = data * 2
print(result)

Vectorised operations are usually shorter, easier to read, and — as you'll now see directly — significantly faster.

### Seeing the difference for real

Let's measure it, on a much larger dataset than either version above, so the difference is obvious rather than too fast to notice:


In [ ]:
import time

n = 2_000_000
data_list = list(range(n))

start = time.time()
result = [0] * n
for i in range(len(data_list)):
    result[i] = data_list[i] * 2
loop_time = time.time() - start

print(f"Loop time: {loop_time:.4f} seconds")

In [ ]:
data_array = np.arange(n)

start = time.time()
result2 = data_array * 2
vector_time = time.time() - start

print(f"Vectorised time: {vector_time:.6f} seconds")
print(f"Speedup: {loop_time / vector_time:.1f}x faster")

Your exact numbers will differ depending on the machine you run this on, but expect the vectorised version to be dramatically faster — often by a factor of 10, 50, or even more. This isn't a minor style preference; at HPC scale, this kind of difference can be the difference between an analysis finishing in seconds versus hours.

> **A notebook-specific tool worth knowing: `%timeit`.** Jupyter provides a built-in "magic" command that times a line of code for you automatically, running it multiple times for a more reliable average. Try it on each version:


In [ ]:
%timeit [x * 2 for x in data_list]

In [ ]:
%timeit data_array * 2

`%timeit` automatically picks a sensible number of repeats and reports the average and variability — much more convenient than wrapping everything in `time.time()` calls yourself, and it's specific to notebooks (it won't work in a plain `.py` script).

## Reduce File Input and Output

Reading and writing files is relatively slow compared to working with data already in memory (recall Lesson 6). Good practice includes:

- reading files once where possible;
- processing multiple values together;
- avoiding unnecessary temporary files;
- writing results only when required.

Efficient file handling becomes increasingly important when processing large datasets.

## Working with Large Datasets

Large scientific datasets require careful memory management. Consider:

- loading only the required columns from a dataset;
- avoiding unnecessary copies of arrays;
- processing data in stages where appropriate;
- selecting suitable data types.

For example, recall `pd.read_csv()` from Lesson 8 — it can load only the columns you actually need via `usecols`, rather than everything in the file:


In [ ]:
import pandas as pd

# A small demonstration file with an "extra" column we don't need
demo = pd.DataFrame({
    "SampleID": ["M1", "M2", "M3"],
    "Temperature": [19.8, 20.5, 21.2],
    "Pressure": [100.5, 101.0, 101.4],
    "Notes": ["ok", "ok", "check"],
})
demo.to_csv("demo_wide.csv", index=False)

# Loading everything:
full = pd.read_csv("demo_wide.csv")
print(full.memory_usage(deep=True).sum(), "bytes (all columns)")

# Loading only what we need:
narrow = pd.read_csv("demo_wide.csv", usecols=["SampleID", "Temperature"])
print(narrow.memory_usage(deep=True).sum(), "bytes (only SampleID and Temperature)")

Small improvements like this can significantly reduce memory usage — and the saving grows with the size of your real dataset, which might have dozens of columns and millions of rows instead of 3 and 4.

## Profiling Before Optimising

Optimisation should be based on evidence rather than guesswork. Python provides several profiling tools, including `time`, `timeit`, and `cProfile` (for profiling a whole program in detail — beyond this lesson's scope, but worth knowing the name of).

A simple example, as you've already used above:


In [ ]:
import time

start = time.time()
# ... analysis code would go here ...
total = sum(range(1_000_000))
print(time.time() - start)

Profiling identifies where a program spends most of its execution time, helping you focus optimisation efforts where they will have the greatest impact — there's little point spending a day speeding up a part of your code that only takes 1% of the total runtime.

## Reproducible Research

Scientific software should produce consistent and repeatable results. Good practice includes:

- documenting software versions;
- recording package dependencies;
- using version control (e.g. git);
- writing clear documentation;
- storing input parameters alongside results.

Python environments (the virtual environments from earlier in this lesson) make it easier to recreate the exact software used in an experiment — which is exactly what the Challenge Exercise at the end of this lesson asks you to practise.

## Python on HPC Systems

Although every HPC service is slightly different, most follow a similar workflow. Typically you will:

1. Connect to the HPC system.
2. Prepare your Python environment.
3. Execute your Python application.
4. Analyse the results.
5. Visualise the output.

Many systems use a workload manager, such as **Slurm**, to execute production jobs. The details of job submission vary between systems and are covered separately in the **Introduction to Slurm** course.


## Worked Examples

### Example 1: Creating a Virtual Environment

Creating a new virtual environment for a research project and installing the required packages is a **terminal task** (see above) — there's nothing to run in this notebook for this example. In your own terminal, for a new project, you'd typically run:

```bash
python -m venv research_env
source research_env/bin/activate
pip install numpy pandas matplotlib
```

### Example 2: Improving Performance

This is the loop-vs-vectorised comparison you already ran above — replacing an explicit Python loop with a vectorised NumPy operation, and comparing execution times with `time.time()` (or `%timeit`).

### Example 3: Complete Scientific Workflow

Let's bring together files (Lesson 6), pandas (Lesson 8), NumPy (Lesson 7) and Matplotlib (Lesson 9) into one small end-to-end pipeline. First, a sample dataset:


In [ ]:
import pandas as pd

benchmark_data = pd.DataFrame({
    "Nodes": [1, 2, 4, 8, 16, 32],
    "Cores": [32, 64, 128, 256, 512, 1024],
    "Runtime": [120.5, 65.2, 34.8, 18.9, 10.2, 6.1],
})
benchmark_data.to_csv("hpc_benchmark.csv", index=False)

print("Sample file created: hpc_benchmark.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Read simulation output using pandas
df = pd.read_csv("hpc_benchmark.csv")
print(df)

# 2. Analyse the data with NumPy / pandas
mean_runtime = np.mean(df["Runtime"])
print(f"\nMean runtime across all runs: {mean_runtime:.2f} seconds")

# 3. Create a graph using Matplotlib
plt.plot(df["Cores"], df["Runtime"], marker="o", label="Measured runtime")
plt.title("HPC Benchmark: Runtime vs Core Count")
plt.xlabel("Number of Cores")
plt.ylabel("Runtime (seconds)")
plt.legend()
plt.grid(True)

# 4. Save the processed results (the figure)
plt.savefig("benchmark_scaling.png")
plt.show()

print("\nFigure saved as benchmark_scaling.png")

Six lessons' worth of skills, in one short, readable pipeline: a file on disk, read into a labelled table, summarised numerically, turned into a figure, and saved for a report. This is exactly the shape of a huge number of real research Python scripts.


<div style="background-color:#1B2A4A; padding:16px 20px; border-radius:6px; color:#FFFFFF; font-family:Arial, Helvetica, sans-serif;">
<h2 style="color:#FFFFFF; margin:0;">Quick Check: Reinforcement Questions</h2>
<p style="color:#C7D0E0; margin-bottom:0;">Test your understanding before moving on to the exercises</p>
</div>

Have a go at each question, then click **"Show answer"** to check yourself.

**1. Why is it good practice to use a virtual environment for a research project?**

a) It makes Python execute more quickly.  b) It isolates project dependencies and improves reproducibility.  c) It replaces the operating system.  d) It automatically parallelises your code.

<details><summary>Show answer</summary>b) It isolates project dependencies and improves reproducibility.</details>

**2. Which command creates a new Python virtual environment?**

a) python create  b) python -m venv research_env  c) pip install venv  d) virtual python

<details><summary>Show answer</summary>b) python -m venv research_env</details>

**3. Which approach generally provides better performance for numerical calculations?**

a) Writing many Python loops.  b) Using NumPy vectorised operations.  c) Repeatedly copying arrays.  d) Reading the same file multiple times.

<details><summary>Show answer</summary>b) Using NumPy vectorised operations.</details>

**4. Why should code be profiled before being optimised?**

a) To make it easier to install.  b) To identify where execution time is being spent.  c) To reduce the size of the source code.  d) To improve code formatting.

<details><summary>Show answer</summary>b) To identify where execution time is being spent.</details>

**5. Which practice helps make scientific research reproducible?**

a) Installing packages globally for every project.  b) Recording software versions and package dependencies.  c) Deleting intermediate results.  d) Avoiding documentation.

<details><summary>Show answer</summary>b) Recording software versions and package dependencies.</details>

**6. Why is reducing unnecessary file input and output important?**

a) It changes the file format.  b) File operations are relatively slow, particularly for large datasets.  c) It makes Python easier to learn.  d) It automatically compresses the data.

<details><summary>Show answer</summary>b) File operations are relatively slow, particularly for large datasets.</details>

**7. Which statement best describes Python's role in HPC?**

a) Python has replaced all compiled scientific applications.  b) Python is used only for visualisation.  c) Python supports scientific workflows by preparing data, automating tasks, analysing results and working with optimised scientific libraries.  d) Python can only be used on personal computers.

<details><summary>Show answer</summary>c) Python supports scientific workflows by preparing data, automating tasks, analysing results and working with optimised scientific libraries.</details>


<div style="background-color:#1B2A4A; padding:16px 20px; border-radius:6px; color:#FFFFFF; font-family:Arial, Helvetica, sans-serif;">
<h2 style="color:#FFFFFF; margin:0;">Student Exercises</h2>
<p style="color:#C7D0E0; margin-bottom:0;">Practice what you've learned</p>
</div>

A few of these exercises are terminal-based rather than notebook-based, in keeping with the lesson itself — these are marked clearly.


### Exercise 1 (terminal task)

In a terminal (not this notebook), create a virtual environment called `research_env` and install NumPy, pandas, and Matplotlib into it.

**Checklist:**

- [ ] Open a terminal (in JupyterHub: File → New → Terminal, or your system's terminal application).
- [ ] Run `python -m venv research_env`.
- [ ] Activate it (`source research_env/bin/activate` on Linux/macOS, `research_env\Scripts\activate` on Windows).
- [ ] Run `pip install numpy pandas matplotlib`.
- [ ] Confirm it worked: with the environment still active, run `python -c "import numpy, pandas, matplotlib; print('All good')"`.
- [ ] Run `deactivate` when you're finished.

> **Optional — try it right here, just for the environment-creation step itself.** The cell below actually creates a real virtual environment folder next to this notebook (this part doesn't need network access), so you can see what `python -m venv` produces. It won't *activate* it, for the reasons explained earlier in the lesson — that part still needs a real terminal.


In [ ]:
!python -m venv research_env
!ls research_env

In [ ]:
import os
# This check only makes sense if you ran the optional cell above.
# If you completed Exercise 1 in a real terminal instead, that's just as valid — skip this check.
if os.path.exists("research_env"):
    assert os.path.exists("research_env/pyvenv.cfg"), "research_env exists but doesn't look like a complete virtual environment."
    print("Correct! research_env was created successfully.")
else:
    print("No research_env folder found here — that's fine if you completed this exercise in a real terminal instead.")

### Exercise 2

Below is an explicit-loop version of a calculation, and some sample data. Rewrite it using a **vectorised** NumPy operation instead (store your answer in a variable called `vectorized_result`).


In [ ]:
import numpy as np

data = np.array([1, 2, 3, 4, 5])

# Loop version (for comparison — don't edit this one):
loop_result = np.zeros_like(data)
for i in range(len(data)):
    loop_result[i] = data[i] ** 2
print("Loop version:", loop_result)

# TODO: rewrite the same calculation (squaring each value) as a vectorized_result, without a loop


In [ ]:
assert "vectorized_result" in globals(), "Expected a variable called vectorized_result."
assert np.array_equal(np.array(vectorized_result), data ** 2), (
    f"Expected vectorized_result to equal data ** 2 ({list(data ** 2)}), got {list(vectorized_result)}."
)
print("Correct! Well done.")

**Reflection (no check for this part):** compare the two versions above. Which is easier to read? Which would you rather debug at 2am before a conference deadline? Jot a sentence or two in the markdown cell below if you'd like.


_Your reflection here._

### Exercise 3 (reflection, no code)

Think back to the data analysis exercise from Lesson 8 (filtering a DataFrame, calculating summary statistics). Identify **one** way the program could be made more efficient when working with a much larger dataset (millions of rows, say, rather than a handful).

Write your answer in the markdown cell below, then compare it with the example answer in the "Show answer" toggle underneath.


_Your answer here._

<details><summary>Show an example answer</summary>

There's more than one good answer here — a few possibilities:

- Use `usecols` in `pd.read_csv()` to load only the columns actually needed for the analysis (as demonstrated earlier in this lesson).
- Choose more memory-efficient data types where possible (e.g. a smaller integer type, or a category type for repeated text labels) rather than accepting pandas' defaults.
- Read and process the file in chunks (`pd.read_csv(..., chunksize=...)`) rather than loading everything into memory at once, if the full dataset doesn't comfortably fit in memory.
- Filter or aggregate data as early as possible in the pipeline, rather than carrying unnecessary rows/columns through every subsequent step.

</details>


### Exercise 4: A Complete Workflow

The cell below provides a sample dataset — **run it first** (this is provided for you, not something to edit):


In [ ]:
import pandas as pd

exercise4_data = pd.DataFrame({
    "Nodes": [1, 2, 4, 8, 16],
    "Cores": [16, 32, 64, 128, 256],
    "Runtime": [88.4, 46.1, 24.9, 13.2, 7.5],
})
exercise4_data.to_csv("scaling_results.csv", index=False)

print("Sample file created: scaling_results.csv")

Combine material from Lessons 6–9 to create a simple scientific workflow that:

- reads `scaling_results.csv` into a DataFrame called `df`;
- calculates the mean runtime and prints it;
- creates a graph of `Runtime` against `Cores`, with a title, axis labels, a legend, and a grid;
- saves the graph as `workflow_summary.png`.


In [ ]:
# TODO: read scaling_results.csv into df, print the mean runtime,
# plot Runtime against Cores (with title, labels, legend, grid), and save as workflow_summary.png

workflow_ax = plt.gca()  # leave this line here, after your plot, before plt.show()
plt.show()

In [ ]:
import os
assert "df" in globals(), "Expected a variable called df."
expected = pd.read_csv("scaling_results.csv")
assert list(df["Runtime"]) == list(expected["Runtime"]), "df doesn't match scaling_results.csv — check you read the right file."
assert len(workflow_ax.lines) >= 1, "Expected a plot of Runtime against Cores."
plotted_y = list(workflow_ax.lines[0].get_ydata())
assert plotted_y == list(expected["Runtime"]), "The plotted data doesn't match the Runtime column."
assert workflow_ax.get_title().strip() != "", "Expected a title."
assert workflow_ax.get_xlabel().strip() != "", "Expected an x-axis label."
assert workflow_ax.get_ylabel().strip() != "", "Expected a y-axis label."
assert workflow_ax.get_legend() is not None, "Expected a legend."
assert workflow_ax.xaxis.get_gridlines()[0].get_visible(), "Expected a grid (plt.grid(True))."
assert os.path.exists("workflow_summary.png") and os.path.getsize("workflow_summary.png") > 0, (
    "Expected a non-empty file called workflow_summary.png."
)
print("Correct! Well done — that's the full Lesson 6-9 pipeline in one go.")

### Challenge Exercise: Preparing Your Project to Share

A colleague needs to reproduce your analysis on another HPC system. Preparing a project to be shared normally involves:

- creating a virtual environment (terminal task, covered above);
- documenting the required Python packages;
- organising your files clearly;
- writing brief instructions explaining how to run the analysis.

The second, third and fourth of these, you can actually do right here — using exactly the file-writing skills from Lesson 6. Create two files:

- `requirements.txt`, listing the packages this project needs (one per line: `numpy`, `pandas`, `matplotlib`).
- `README.md`, with a short explanation of what the project does and how to run it (make sure to mention creating/activating the virtual environment, and installing from `requirements.txt`).


In [ ]:
# TODO: write requirements.txt (listing numpy, pandas, matplotlib) and README.md
# (explaining what the project does and how to set it up and run it)


In [ ]:
import os
assert os.path.exists("requirements.txt"), "Expected a file called requirements.txt."
assert os.path.exists("README.md"), "Expected a file called README.md."

with open("requirements.txt") as f:
    req_content = f.read().lower()
for pkg in ("numpy", "pandas", "matplotlib"):
    assert pkg in req_content, f"Expected '{pkg}' to be listed in requirements.txt."

with open("README.md") as f:
    readme_content = f.read().lower()
assert len(readme_content.strip()) > 0, "README.md appears to be empty."
assert "venv" in readme_content or "environment" in readme_content or "pip install" in readme_content, (
    "Your README should mention how to set up the environment (e.g. creating a virtual environment, or pip install)."
)
print("Correct! Well done — a colleague could now reproduce this project from scratch.")

## Summary

In this lesson, you explored how Python is used throughout scientific computing workflows on HPC systems. You learned how to manage Python environments, write more efficient programs, work with large datasets and apply good software engineering practices to improve reproducibility.

Rather than introducing new language features, this lesson demonstrated how the techniques developed throughout the course can be combined to build robust, maintainable and portable scientific applications suitable for use on modern HPC systems.

### Key Takeaways

- Python is widely used throughout modern scientific computing workflows.
- Virtual environments help manage dependencies and improve reproducibility.
- Scientific Python libraries such as NumPy and pandas provide efficient, optimised operations for numerical and tabular data.
- Vectorised operations are generally preferable to explicit Python loops when working with arrays.
- Efficient file handling and memory management become increasingly important as datasets grow.
- Profiling should guide optimisation efforts by identifying genuine performance bottlenecks.
- Good software engineering practices — including documentation, version control and dependency management — are essential for reproducible research.
- The skills developed throughout this course provide a strong foundation for using Python effectively on DiRAC and other UK HPC systems.


<div style="background-color:#1B2A4A; padding:24px 28px; border-radius:6px; color:#FFFFFF; font-family:Arial, Helvetica, sans-serif;">
<h2 style="color:#FFFFFF; margin-top:0;">🎓 Course Completion</h2>
<p style="color:#C7D0E0;">Congratulations — you have completed the <strong>Introduction to Python for High Performance Computing</strong> course.</p>
<p style="color:#C7D0E0;">Over the past ten lessons, you have learned how to:</p>
<ul style="color:#C7D0E0;">
<li>Write clear, well-structured Python programs.</li>
<li>Work with variables, collections, loops and functions.</li>
<li>Read and write files.</li>
<li>Perform numerical computing with NumPy.</li>
<li>Analyse structured datasets with pandas.</li>
<li>Create publication-quality scientific figures using Matplotlib.</li>
<li>Apply Python effectively within modern HPC workflows.</li>
</ul>
<p style="color:#C7D0E0; margin-bottom:0;">You now have the knowledge and practical skills to begin developing Python applications that support computational research on DiRAC and other HPC services.</p>
</div>

### Next Steps

To continue developing your HPC programming skills, you may wish to explore:

- **Introduction to Slurm** — Running applications efficiently on HPC systems.
- **Introduction to MPI** — Parallel programming across multiple processes.
- **Introduction to OpenMP** — Shared-memory parallel programming.
- **HIP Programming for AMD GPUs** — Accelerating applications on modern GPU architectures.
- **Performance Portability with Kokkos** — Writing portable applications for diverse HPC hardware.

These courses build directly on the Python skills you have developed here and provide a pathway towards writing scalable, high-performance scientific applications.


<div style="background-color:#1B2A4A; padding:18px 22px; border-radius:6px; color:#FFFFFF; font-family:Arial, Helvetica, sans-serif; text-align:center;">
<p style="margin:0; color:#FFFFFF; font-weight:bold;">DiRAC Training Academy</p>
<p style="margin:4px 0 0 0; color:#C7D0E0; font-size:13px;">dirac.ac.uk</p>
</div>
